Here I download data from API ISTAT and save it locally so I don't have to call the API everytime

In [2]:
import os
import requests
import pandas as pd

In [3]:
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}
file_path = "../data/raw/istat_road_accidents.csv"

In [4]:
if os.path.exists(file_path):
    print("file already downloaded")
else: 
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print("file downloaded")
    else:
        print("Error:", response.status_code)

file already downloaded


First look at the data

In [5]:
df = pd.read_csv(file_path)
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.shape


(573552, 16)

The column REF_AREA don't have anymore the initial zeros shown in the csv file because it's read as a number.

In [7]:
df = pd.read_csv(file_path, dtype={"REF_AREA": str})
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,001001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,001001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,001001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,001001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,001001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.dtypes

DATAFLOW                str
FREQ                    str
REF_AREA                str
DATA_TYPE               str
RESULT                  str
TIME_PERIOD           int64
OBS_VALUE             int64
OBS_STATUS          float64
NOTE_DS             float64
NOTE_REF_AREA       float64
NOTE_DATA_TYPE      float64
NOTE_RESULT         float64
NOTE_TIME_PERIOD    float64
BASE_PER            float64
UNIT_MEAS           float64
UNIT_MULT           float64
dtype: object

Controlling the columns

In [9]:
df.isnull().sum()

DATAFLOW                 0
FREQ                     0
REF_AREA                 0
DATA_TYPE                0
RESULT                   0
TIME_PERIOD              0
OBS_VALUE                0
OBS_STATUS          573552
NOTE_DS             573552
NOTE_REF_AREA       573552
NOTE_DATA_TYPE      573552
NOTE_RESULT         573552
NOTE_TIME_PERIOD    573552
BASE_PER            573552
UNIT_MEAS           573552
UNIT_MULT           573552
dtype: int64

In [10]:
df["DATA_TYPE"].value_counts()

DATA_TYPE
KILLINJ    382368
ROADACC    191184
Name: count, dtype: int64

In [11]:
df["RESULT"].value_counts()

RESULT
F    191184
M    191184
9    191184
Name: count, dtype: int64

In [12]:
df["TIME_PERIOD"].unique()

array([2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011,
       2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022,
       2023, 2024])

I noticed that column KILLINJ has twice the number of rows as column ROADACC and F,M,9 have the same amount of rows. I want to check the combination between DATA_TYPE and RESULT.

In [13]:
df.groupby(["DATA_TYPE", "RESULT"]).size()

DATA_TYPE  RESULT
KILLINJ    F         191184
           M         191184
ROADACC    9         191184
dtype: int64

So 9 refers only to ROADACC

In [14]:
df[(df["REF_AREA"] == "001001") & (df["TIME_PERIOD"] == 2023)]

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
22,IT1:41_983(1.0),A,001001,KILLINJ,F,2023,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,IT1:41_983(1.0),A,001001,KILLINJ,M,2023,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70,IT1:41_983(1.0),A,001001,ROADACC,9,2023,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df[(df["REF_AREA"] == "001002") & (df["TIME_PERIOD"] == 2023)]

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
94,IT1:41_983(1.0),A,001002,KILLINJ,F,2023,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
118,IT1:41_983(1.0),A,001002,KILLINJ,M,2023,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142,IT1:41_983(1.0),A,001002,ROADACC,9,2023,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


I see that for each municipality it gives alwas three rows such as KILLINJ F, KILLINJ M, ROADACC.
So KILLINJ refers to kills (M=morti), and injuries (F=feriti); ROADACC is road accidents and I think 9 does not give any information but it can be considered as null or without a meaning because it refers only to road accidents, not a person.
I notice also that there are more injuries than kills and also than accidents.

Now I want to see if F AND M refers to "feriti" and "morti": if the sum of M and F rows is equal to the official ISTAT numbers. If it's half of the people it refers to "maschi". I want also to check ROADACC.

In [16]:
df_2023 = df[df["TIME_PERIOD"] == 2023]
df_2023.groupby(["DATA_TYPE", "RESULT"])["OBS_VALUE"].sum()

DATA_TYPE  RESULT
KILLINJ    F         224634
           M           3039
ROADACC    9         166525
Name: OBS_VALUE, dtype: int64

The 2023 totals match the official ISTAT data so my interpretation is confirmed.

Web scraping on SITUAS data:

In [19]:
from bs4 import BeautifulSoup
page = requests.get("https://situas.istat.it/web/")
soup = BeautifulSoup(page.content, "html.parser") 
soup.find_all("table")

               

[]

In [20]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="it">
 <head>
  <meta charset="utf-8"/>
  <link href="./favicon.ico" rel="icon"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <meta content="#000000" name="theme-color"/>
  <meta content="ISTAT - SITUAS" name="description"/>
  <script data-dtconfig="rid=RID_1458574583|rpid=263464495|domain=istat.it|reportUrl=/web/rb_a34aa096-4709-4067-be2f-3b68462892d0|app=5f9422794d4b081c|cuc=lwetihge|owasp=1|mel=100000|expw=1|featureHash=ICA7NVfqrux|dpvc=1|lastModification=1790601578166|tp=500,50,0|rdnt=1|uxrgce=1|srbbv=2|agentUri=/web/ruxitagentjs_ICA7NVfqrux_10345260810170330.js" src="/web/ruxitagentjs_ICA7NVfqrux_10345260810170330.js" type="text/javascript">
  </script>
  <link href="./logo192.png" rel="apple-touch-icon"/>
  <link href="./manifest.json" rel="manifest"/>
  <script>
   var _paq=window._paq=window._paq||[];_paq.push(["trackPageView"]),_paq.push(["enableLinkTracking"]),function(){var a="https://analytics.istat.it/";_paq.push(["se

BeautifulSoup doesn't find a table. After printing the HTML, the page is almost empty and it needs Javascript. So I'll use the browser devtools to see from where page are the data and call it with request.I use the most recent year, 2024, so we'll have the same period for population and accidents.

In [21]:
situas_url = "https://situas.istat.it/ShibO2Module/api/Report/Spool/2024-12-31/74?&PageNum=1&PageSize=10000"
situas_path = "../data/raw/situas_municipalities_2024.csv"

In [24]:
if os.path.exists(situas_path):
    print("file already downloaded")
else:
    response = requests.post(situas_url)
    if response.status_code == 200:
        data = response.json()
        df_situas = pd.DataFrame(data["body"])
        df_situas.to_csv(situas_path, index=False)
        print("file downloaded")
    else: 
        print("error:", response.status_code)

file already downloaded


In [25]:
df_situas = pd.read_csv(situas_path, dtype={"PRO_COM_T": str})
df_situas.head()

,COD_RIP,COD_REG,COD_PROV_STORICO,COD_UTS,PRO_COM_T,PRO_COM,COMUNE,SIGLA_AUTOMOBILISTICA,CC_UTS,CC_REG,POP_LEG,ANNO_CENSIMENTO,AREA_KMQ,ANNO_AREA,POP_RES,ANNO_POP_RES,COMUNE_A
0,1,1,1,201,001001,1001,Agliè,TO,0,0,2562,2021,13.1463,2024,2585,2024,NaN
1,1,1,1,201,001002,1002,Airasca,TO,0,0,3660,2021,15.7393,2024,3695,2024,NaN
2,1,1,1,201,001003,1003,Ala di Stura,TO,0,0,467,2021,46.3316,2024,463,2024,NaN
3,1,1,1,201,001004,1004,Albiano d'Ivrea,TO,0,0,1637,2021,11.7397,2024,1624,2024,NaN
4,1,1,1,201,001006,1006,Almese,TO,0,0,6331,2021,17.8741,2024,6297,2024,NaN


In [26]:
df_situas.shape

(7896, 17)

In [27]:
df_situas.isnull().sum()


COD_RIP                     0
COD_REG                     0
COD_PROV_STORICO            0
COD_UTS                     0
PRO_COM_T                   0
PRO_COM                     0
COMUNE                      1
SIGLA_AUTOMOBILISTICA      92
CC_UTS                      0
CC_REG                      0
POP_LEG                     0
ANNO_CENSIMENTO             0
AREA_KMQ                    0
ANNO_AREA                   0
POP_RES                     0
ANNO_POP_RES                0
COMUNE_A                 7772
dtype: int64

In [28]:
df_situas.describe()

,COD_RIP,COD_REG,COD_PROV_STORICO,COD_UTS,PRO_COM,CC_UTS,CC_REG,POP_LEG,ANNO_CENSIMENTO,AREA_KMQ,ANNO_AREA,POP_RES,ANNO_POP_RES
count,7896.000000,7896.000000,7896.000000,7896.000000,7896.000000,7896.000000,7896.000000,7.896000e+03,7896.0,7896.000000,7896.0,7.896000e+03,7896.0
mean,2.487335,8.664767,45.178951,77.296479,45241.979863,0.014184,0.002533,7.475954e+03,2021.0,38.261090,2024.0,7.464978e+03,2024.0
std,1.429214,6.390636,32.663820,79.326172,32641.735438,0.118258,0.050268,4.152719e+04,0.0,50.823422,0.0,4.154247e+04,0.0
min,1.000000,1.000000,1.000000,2.000000,1001.000000,0.000000,0.000000,3.100000e+01,2021.0,0.122200,2024.0,3.200000e+01,2024.0
25%,1.000000,3.000000,16.000000,19.000000,16147.750000,0.000000,0.000000,9.880000e+02,2021.0,11.470625,2024.0,9.690000e+02,2024.0
50%,2.000000,7.000000,40.000000,57.000000,40043.500000,0.000000,0.000000,2.411500e+03,2021.0,22.456900,2024.0,2.383000e+03,2024.0
75%,4.000000,15.000000,73.000000,94.000000,73007.250000,0.000000,0.000000,6.240000e+03,2021.0,44.756775,2024.0,6.239250e+03,2024.0
max,5.000000,20.000000,111.000000,292.000000,111107.000000,1.000000,1.000000,2.749031e+06,2021.0,1288.187500,2024.0,2.747290e+06,2024.0


In [29]:
df_situas.sort_values("POP_RES") [["COMUNE", "POP_RES"]].head()

,COMUNE,POP_RES
7408,Morterone,32
1668,Pedesina,37
511,Briga Alta,40
590,Macra,42
119,Ingria,44


In [30]:
df_situas.sort_values("POP_RES")[["COMUNE", "POP_RES"]].tail()

,COMUNE,POP_RES
6595,Palermo,628693
267,Torino,853196
5104,Napoli,909013
1771,Milano,1365698
4719,Roma,2747290
